# Poultry Health Monitoring — ML Subsystem Training (Section 3.9)

YOLO11n transfer learning for bird detection, following the dissertation's
machine-learning development pipeline (Fig. 3.11): dataset acquisition →
cleaning → leakage-safe splitting → transfer learning → evaluation → export.

**Runtime → Change runtime type → T4 GPU** before running.

Data: Broiler-Net material (Zarrat Ehsan & Mohtavipour, 2024), Apache-2.0,
github.com/TaherehZarratEhsan/Chicken-Behavior-Analysis

In [ ]:
# --- 1. Training environment (screenshot this cell's output for Fig 3.14) ---
!nvidia-smi
import sys, platform
print('Python', sys.version)
print('Platform', platform.platform())

In [ ]:
# --- 2. Install dependencies ---
%pip -q install ultralytics
import ultralytics
ultralytics.checks()

In [ ]:
# --- 3. Dataset acquisition (3.9.2) ---
!git clone --depth 1 https://github.com/TaherehZarratEhsan/Chicken-Behavior-Analysis.git /content/cba

In [ ]:
# --- 4. Cleaning, leakage-safe split, tiling (3.9.6-3.9.8) ---
# Converts Pascal VOC XML to YOLO format, splits by video segment so frames
# from one video never cross splits, and tiles 1920x1080 frames into 640px crops.
import hashlib, json, random, xml.etree.ElementTree as ET
from pathlib import Path
import cv2
random.seed(42)
SRC, OUT = Path('/content/cba'), Path('/content/dataset')
TILE, STRIDE, MIN_BOX = 640, 480, 8
frames, seen = [], set()
for xf in sorted(SRC.glob('xml/*.xml')):
    r = ET.parse(xf).getroot()
    seg = Path(r.findtext('path').replace('\\', '/')).parent.name
    cands = list((SRC/'img').glob(xf.stem + '.*'))
    if not cands: continue
    img = cv2.imread(str(cands[0]))
    if img is None: continue
    h = hashlib.md5(img.tobytes()).hexdigest()
    if h in seen: continue
    seen.add(h)
    boxes = [tuple(int(b.findtext(t)) for t in ('xmin','ymin','xmax','ymax'))
             for b in (o.find('bndbox') for o in r.findall('object'))]
    frames.append((cands[0], seg, [b for b in boxes if b[2]>b[0] and b[3]>b[1]]))
segs = sorted({f[1] for f in frames}); random.shuffle(segs)
val = set(segs[:2])
for s in ('images/train','images/val','labels/train','labels/val'):
    (OUT/s).mkdir(parents=True, exist_ok=True)
for p, seg, boxes in frames:
    split = 'val' if seg in val else 'train'
    img = cv2.imread(str(p)); H, W = img.shape[:2]; tid = 0
    ys = sorted(set(list(range(0, H-TILE+1, STRIDE)) + [H-TILE]))
    xs = sorted(set(list(range(0, W-TILE+1, STRIDE)) + [W-TILE]))
    for ty in ys:
        for tx in xs:
            kept = []
            for x1,y1,x2,y2 in boxes:
                cx, cy = (x1+x2)/2, (y1+y2)/2
                if not (tx <= cx < tx+TILE and ty <= cy < ty+TILE): continue
                nx1,ny1 = max(x1,tx)-tx, max(y1,ty)-ty
                nx2,ny2 = min(x2,tx+TILE)-tx, min(y2,ty+TILE)-ty
                if nx2-nx1 >= MIN_BOX and ny2-ny1 >= MIN_BOX: kept.append((nx1,ny1,nx2,ny2))
            if not kept: continue
            base = f'{p.stem}_t{tid:02d}'
            cv2.imwrite(str(OUT/f'images/{split}/{base}.jpg'), img[ty:ty+TILE, tx:tx+TILE])
            with open(OUT/f'labels/{split}/{base}.txt', 'w') as fh:
                for nx1,ny1,nx2,ny2 in kept:
                    fh.write(f'0 {(nx1+nx2)/2/TILE:.6f} {(ny1+ny2)/2/TILE:.6f} '
                             f'{(nx2-nx1)/TILE:.6f} {(ny2-ny1)/TILE:.6f}\n')
            tid += 1
(OUT/'data.yaml').write_text(f'path: {OUT}\ntrain: images/train\nval: images/val\n\nnames:\n  0: chicken\n')
print('tiles:', len(list((OUT/'images/train').glob('*'))), 'train /',
      len(list((OUT/'images/val').glob('*'))), 'val')

In [ ]:
# --- 5. Transfer learning (3.9.11-3.9.12) ---
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
results = model.train(data='/content/dataset/data.yaml', epochs=60, imgsz=640,
                      batch=16, seed=42, patience=15, project='runs', name='detect_v1')

In [ ]:
# --- 6. Evaluation artefacts (3.9.14) ---
# results.png (loss curves), PR_curve.png, confusion_matrix.png land in runs/detect_v1/
from IPython.display import Image, display
for f in ('results.png', 'PR_curve.png', 'confusion_matrix.png'):
    display(Image(f'runs/detect_v1/{f}'))

In [ ]:
# --- 7. Model export for the Raspberry Pi (3.9.15) ---
best = YOLO('runs/detect_v1/weights/best.pt')
best.export(format='onnx', imgsz=640)
best.export(format='tflite', imgsz=640)